# Session 1: teach a network to compute n mod 6

**What you will see.** We show a small neural network thousands of numbers together with their remainders mod 6. It gets no rules, only examples. Its error does not fall smoothly. It falls, stalls, and falls again. In session 2 we open the network up and find out what it knew at each stage.

**How to use this notebook.** Run the cells in order with **Shift + Enter**. Before each training run there is a question. Say your guess to your neighbor before you run the cell.

In [ ]:
# Setup: find the course files, downloading them first if this is Colab.
import os, sys, shutil
REPO = "https://github.com/icarm/SACNAS-2026"
if "google.colab" in sys.modules and not os.path.exists("/content/course/mlmath.py"):
    shutil.rmtree("/content/course", ignore_errors=True)
    if os.system(f"git clone -q --depth 1 {REPO} /content/course") != 0:
        raise RuntimeError("Could not download the course files. Check the wifi, then run this cell again.")
for folder in ["/content/course", ".", ".."]:
    if os.path.exists(os.path.join(folder, "mlmath.py")):
        os.chdir(folder)
        break
else:
    raise RuntimeError("Could not find the course files. Open this notebook from the ml4math-mini-course folder.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Ready.")

## 1. The game, by hand

The remainder of n after dividing by 6 is written **n mod 6**. For example 4712 mod 6 = 2, because 4712 = 6 × 785 + 2.

Try these without dividing: **1234, 9000, 5555, 2026**.

Most people end up using two shortcuts.

- **Even or odd?** Look at the last digit only.
- **Remainder mod 3?** Add up the digits. For 4712: 4 + 7 + 1 + 2 = 14, and 14 mod 3 = 2.

Together these pin down n mod 6. That is the Chinese Remainder Theorem: knowing n mod 2 and n mod 3 is the same as knowing n mod 6.

Keep these two shortcuts in mind. We will look for them inside the network.

## 2. The data

This cell builds every number from 0000 to 9999 and labels each with n mod 6. We write numbers with leading zeros so every number has exactly four digits.

In [ ]:
import torch
import torch.nn as nn
import mlmath

data = mlmath.make_data(modulus=6)

for n in [0, 7, 1234, 4712, 9999]:
    print(f"{n:04d}  ->  {n % 6}")

We split the numbers at random into two halves. The network **trains** on one half. The other half is the **test** set, which it never sees during training.

Why: a network could score 100% on the training half by memorizing it, the way you could memorize an answer key. The test half tells us whether it learned a rule that works on new numbers.

In [ ]:
print("training numbers:", len(data.train))
print("test numbers:    ", len(data.test))
print("a few training numbers:", sorted(data.train.tolist())[:8])
print("a few test numbers:    ", sorted(data.test.tolist())[:8])

## 3. How do you show a number to a network?

A network takes a list of numbers as input. There is more than one way to turn 4712 into a list, and the choice matters a great deal. Here are three.

In [ ]:
n = torch.tensor([4712])
print("1. Raw value (1 input):       ", [round(v, 4) for v in mlmath.encode_raw(n)[0].tolist()])
print("2. Digit values (4 inputs):   ", [round(v, 2) for v in mlmath.encode_digits(n)[0].tolist()])
print("3. One-hot digits (40 inputs):")
for place, row in zip(["thousands", "hundreds", "tens", "ones"], mlmath.encode_onehot(n)[0].reshape(4, 10)):
    print(f"   {place:9s}", [int(v) for v in row.tolist()])

In the **one-hot** encoding each digit gets ten switches, and exactly one is on. The digit 7 is "switch number 7 is on". Nothing in this encoding says that 7 is bigger than 2 or close to 8.

We will try all three, starting with the simplest.

## 4. The network and the training loop

The network is as plain as they come: the inputs, then one layer of 512 hidden units, then 6 outputs, one score for each possible answer 0 to 5. The highest score is the network's answer.

Training repeats four steps.

1. Show the network a small batch of training numbers.
2. Measure how wrong its answers are. That measurement is the **loss**. Lower is better.
3. Work out which way to nudge each connection strength to make the loss a little smaller.
4. Nudge them. Repeat.

One pass through all the training numbers is called an **epoch**. Read the loop below once before running it. The four steps are marked. Running this cell only defines the loop; nothing trains yet.

In [ ]:
def train(encode, data, width=512, lr=0.1, epochs=400, batch_size=64, seed=0):
    torch.manual_seed(seed)                                    # same starting point for everyone
    X, y = encode(data.numbers, data.n_digits), data.labels
    model = nn.Sequential(nn.Linear(X.shape[1], width), nn.ReLU(), nn.Linear(width, data.modulus))
    model.encode = encode
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(epochs + 1):
        history.append(mlmath.evaluate(model, X, y, data, epoch))    # measure progress
        if epoch % 10 == 0 or epoch == epochs:
            mlmath.live_plot(history, data.modulus)                  # redraw the picture
        if epoch == epochs:
            break

        order = data.train[torch.randperm(len(data.train))]          # shuffle the training numbers
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            loss = loss_fn(model(X[batch]), y[batch])                # steps 1 and 2: answer, measure
            optimizer.zero_grad()
            loss.backward()                                          # step 3: which way to nudge
            optimizer.step()                                         # step 4: nudge

    return model, history

## 5. Experiment 1: the raw number

**Predict first.** The network gets one input, n / 10000. Guessing would be right 1 time in 6, about 17%. After 150 epochs, will it beat guessing?

How to read the plot: the top panel is the loss. The bottom panel is accuracy on the test numbers. Blue is "exactly right", orange is "got the parity right", green is "got the remainder mod 3 right". Once parity is perfect, blue and green lie on top of each other, because the only way left to be wrong is mod 3.

In [ ]:
model_raw, history_raw = train(mlmath.encode_raw, data, epochs=150)
mlmath.summarize(history_raw)

**Discuss.** Why can it not learn this? Think about 4711, 4712, 4713. Their inputs are 0.4711, 0.4712, 0.4713. What are their labels?

A network like this one turns nearby inputs into nearby outputs. Here nearby numbers have unrelated answers. To get even parity right it would need a function that flips back and forth 10,000 times.

## 6. Experiment 2: four digit values

Now the network sees the four digits as four numbers. The last digit is right there in the input, and parity depends only on the last digit.

**Predict first.** Will it at least get parity?

In [ ]:
model_digits, history_digits = train(mlmath.encode_digits, data, epochs=150)
mlmath.summarize(history_digits)

**Discuss.** The information is all there. Why is it still stuck?

This encoding tells the network that 3 lies between 2 and 4. That is true for size and useless for divisibility: 2 and 4 are even and 3 is not. The network has to fight the encoding to treat neighbors differently.

## 7. Experiment 3: one-hot digits

Same network, same training loop, same settings. Only the encoding changes. This run is longer, 400 epochs, and takes about a minute.

**Predict first.** Sketch the loss curve you expect. A smooth slide? One cliff? Something else?

In [ ]:
model, history = train(mlmath.encode_onehot, data, epochs=400)
mlmath.summarize(history)

## 8. Read the plot

The epoch axis is stretched so the early epochs are not squashed. Look at the loss panel first.

- The loss starts at the dotted line marked **ln 6**. Why that value? At the start the network is guessing among 6 answers.
- It drops fast and then sits on a flat stretch. Which dotted line is it sitting on? What does that say about how many answers it is still choosing among?
- Now look at the accuracy panel during the flat stretch. Which curve is already at 100%? Which is still at the guessing level?
- During the flat stretch the dashed curve (training numbers) rises above the solid one (test numbers). What is the network doing?

The cell below prints the same information as numbers at a few moments.

In [ ]:
print("epoch   loss   exactly right   parity right   right mod 3")
for epoch in [0, 2, 10, 30, 60, 100, 200, 400]:
    row = history[epoch]
    print(f"{epoch:5d}  {row['test_loss']:5.2f}  {row['test_acc']:13.0%}  {row['mod2_acc']:13.0%}  {row['mod3_acc']:12.0%}")

## Questions for the break

1. At epoch 30 the network scores about one in three. Is it guessing at random, or does it know something? How could you tell the difference?
2. Between epoch 30 and epoch 400 it learns something new. In your own words, what do you think it learned?
3. We never told the network about digit sums or parity. If it found those rules, where inside it would you look for them?